# Terry TinyLLM Notebook

This notebook inlines the Terry training project so it can run standalone without importing local project modules. Run the cells from top to bottom.


# 1. Environment & Setup

Install dependencies and import required libraries.

In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import time
from collections import deque
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace
from typing import Any, Callable, Iterable, Iterator, Optional, Sequence

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset, IterableDataset

# Install dependencies if needed (uncomment if running in Colab)
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 2. General Configuration

Define model and training configurations.

In [ ]:
@dataclass
class ModelConfig:
    d_model: int = 256
    n_layers: int = 8
    n_heads: int = 8
    ffn_mult: int = 4
    max_seq_len: int = 1024
    dropout_rate: float = 0.0


@dataclass
class TrainConfig:
    lr: float = 1e-4
    weight_decay: float = 0.01
    batch_size: int = 4
    grad_accum: int = 8
    device: str = 'auto'
    warmup_steps: int = 2000
    total_steps: int = 17000
    seed: int = 42
    train_source_path: str = 'src/terry_daily_chat_train.jsonl'
    valid_source_path: str = 'src/terry_daily_chat_valid.jsonl'
    train_tokens_path: str = 'src/processed/terry_train_tokens.txt'
    valid_tokens_path: str = 'src/processed/terry_valid_tokens.txt'
    tokenizer_dir: str = 'tokenizer/terry_byte'
    train_samples: int = 60_000
    valid_samples: int = 2_000


# Tokenizer constants
PAD_TOKEN = '<|pad|>'
BOS_TOKEN = '<|im_start|>'
EOS_TOKEN = '<|im_end|>'

PAD_TOKEN_ID = 0
BOS_TOKEN_ID = 1
EOS_TOKEN_ID = 2

BYTE_OFFSET = 3
VOCAB_SIZE = BYTE_OFFSET + 256

DEFAULT_TOKENIZER_DIR = Path('tokenizer/terry_byte')
DEFAULT_TRAIN_PATH = Path('src/terry_daily_chat_train.jsonl')
DEFAULT_VALID_PATH = Path('src/terry_daily_chat_valid.jsonl')
DEFAULT_PROCESSED_DIR = Path('src/processed')
DEFAULT_TRAIN_TOKENS = DEFAULT_PROCESSED_DIR / 'terry_train_tokens.txt'
DEFAULT_VALID_TOKENS = DEFAULT_PROCESSED_DIR / 'terry_valid_tokens.txt'

# 3. Data Pipeline

Define tokenizer, dataset generation, and data loading utilities.

In [ ]:
class ByteTokenizer:
    def __init__(self, model_max_length: int | None = None):
        self.pad_token = PAD_TOKEN
        self.bos_token = BOS_TOKEN
        self.eos_token = EOS_TOKEN
        self.pad_token_id = PAD_TOKEN_ID
        self.bos_token_id = BOS_TOKEN_ID
        self.eos_token_id = EOS_TOKEN_ID
        self.model_max_length = model_max_length or max(ModelConfig().max_seq_len, 1_000_000)

    def __len__(self) -> int:
        return VOCAB_SIZE

    def encode(
        self,
        text: str,
        add_special_tokens: bool = False,
        return_tensors: str | None = None,
    ):
        if not isinstance(text, str):
            raise TypeError(f'Expected text to be str, got {type(text)!r}')

        ids = [BYTE_OFFSET + value for value in text.encode('utf-8')]

        if add_special_tokens:
            ids = [self.bos_token_id, *ids, self.eos_token_id]

        if return_tensors == 'pt':
            return torch.tensor([ids], dtype=torch.long)

        return ids

    def decode(
        self,
        ids: Sequence[int] | torch.Tensor,
        skip_special_tokens: bool = False,
        clean_up_tokenization_spaces: bool = True,
    ) -> str:
        del clean_up_tokenization_spaces

        flat_ids = self._flatten_ids(ids)
        pieces: list[str] = []
        byte_buffer = bytearray()

        for token_id in flat_ids:
            if token_id >= BYTE_OFFSET:
                byte_buffer.append(token_id - BYTE_OFFSET)
                continue

            if byte_buffer:
                pieces.append(byte_buffer.decode('utf-8', errors='ignore'))
                byte_buffer.clear()

            if skip_special_tokens:
                continue

            if token_id == self.pad_token_id:
                pieces.append(self.pad_token)
            elif token_id == self.bos_token_id:
                pieces.append(self.bos_token)
            elif token_id == self.eos_token_id:
                pieces.append(self.eos_token)

        if byte_buffer:
            pieces.append(byte_buffer.decode('utf-8', errors='ignore'))

        return ''.join(pieces)

    def __call__(
        self,
        text: str,
        add_special_tokens: bool = False,
        truncation: bool = False,
        return_tensors: str | None = None,
    ):
        del truncation
        input_ids = self.encode(
            text,
            add_special_tokens=add_special_tokens,
            return_tensors=return_tensors,
        )
        return SimpleNamespace(input_ids=input_ids)

    def convert_ids_to_tokens(self, ids: Iterable[int]) -> list[str]:
        tokens = []
        for token_id in ids:
            if token_id == self.pad_token_id:
                tokens.append(self.pad_token)
            elif token_id == self.bos_token_id:
                tokens.append(self.bos_token)
            elif token_id == self.eos_token_id:
                tokens.append(self.eos_token)
            else:
                tokens.append(bytes([token_id - BYTE_OFFSET]).decode('utf-8', errors='ignore'))
        return tokens

    def save_pretrained(self, save_directory: str | Path):
        save_path = Path(save_directory)
        save_path.mkdir(parents=True, exist_ok=True)

        config = {
            'tokenizer_type': 'byte',
            'pad_token': self.pad_token,
            'bos_token': self.bos_token,
            'eos_token': self.eos_token,
            'pad_token_id': self.pad_token_id,
            'bos_token_id': self.bos_token_id,
            'eos_token_id': self.eos_token_id,
            'byte_offset': BYTE_OFFSET,
            'vocab_size': VOCAB_SIZE,
            'model_max_length': self.model_max_length,
        }

        with (save_path / 'tokenizer_config.json').open('w', encoding='utf-8') as handle:
            json.dump(config, handle, indent=2)

        return (str(save_path),)

    @classmethod
    def from_pretrained(cls, load_directory: str | Path) -> 'ByteTokenizer':
        config_path = Path(load_directory) / 'tokenizer_config.json'
        if not config_path.exists():
            raise FileNotFoundError(f'Tokenizer config not found at {config_path}')

        with config_path.open('r', encoding='utf-8') as handle:
            config = json.load(handle)

        tokenizer = cls(model_max_length=config.get('model_max_length'))
        expected = {
            'pad_token_id': PAD_TOKEN_ID,
            'bos_token_id': BOS_TOKEN_ID,
            'eos_token_id': EOS_TOKEN_ID,
        }
        for key, value in expected.items():
            if config.get(key) != value:
                raise ValueError(f'Unsupported tokenizer config: {key}={config.get(key)}')

        return tokenizer

    @staticmethod
    def _flatten_ids(ids: Sequence[int] | torch.Tensor) -> list[int]:
        if isinstance(ids, torch.Tensor):
            return ids.detach().cpu().view(-1).tolist()

        if ids and isinstance(ids[0], (list, tuple)):
            flat: list[int] = []
            for item in ids:
                flat.extend(item)
            return flat

        return list(ids)


def load_tokenizer(tokenizer_path: str | Path | None = None) -> ByteTokenizer:
    if tokenizer_path is not None:
        path = Path(tokenizer_path)
        if (path / 'tokenizer_config.json').exists():
            return ByteTokenizer.from_pretrained(path)

    if (DEFAULT_TOKENIZER_DIR / 'tokenizer_config.json').exists():
        return ByteTokenizer.from_pretrained(DEFAULT_TOKENIZER_DIR)

    return ByteTokenizer()

In [ ]:
def normalize_text(text: str) -> str:
    return ' '.join(text.strip().lower().split())


def message(role: str, content: str) -> dict[str, str]:
    return {'role': role, 'content': normalize_text(content)}


class TerryDatasetGenerator:
    def __init__(self, seed: int = 42):
        self.rng = random.Random(seed)
        self.generators: list[Callable[[], dict[str, object]]] = [
            self.greeting_chat,
            self.meal_chat,
            self.sleep_chat,
            self.room_observation_chat,
            self.memory_chat,
            self.learning_word_chat,
            self.feelings_chat,
            self.counting_chat,
            self.cleanup_chat,
            self.weather_guess_chat,
            self.outside_limit_chat,
            self.story_chat,
            self.choice_chat,
            self.waiting_chat,
            self.noise_chat,
            self.small_mistake_chat,
            self.plan_chat,
            self.color_chat,
            self.object_compare_chat,
            self.ownership_chat,
            self.play_chat,
            self.bedtime_chat,
            self.curiosity_chat,
            self.smell_chat,
        ]

        self.rooms = [
            'kitchen',
            'hall',
            'sofa corner',
            'desk area',
            'bedroom',
            'door side',
            'window spot',
            'lamp table',
        ]
        self.items = [
            'cup',
            'spoon',
            'sock',
            'pillow',
            'book',
            'remote',
            'bowl',
            'key',
            'blanket',
            'notebook',
            'pen',
            'slipper',
            'plate',
            'phone',
            'towel',
            'backpack',
        ]
        self.foods = [
            'toast',
            'rice',
            'noodles',
            'apple slices',
            'soup',
            'eggs',
            'banana',
            'bread',
            'cookies',
            'dumplings',
        ]
        self.drinks = ['water', 'milk', 'tea', 'juice', 'warm cocoa']
        self.colors = ['red', 'blue', 'green', 'yellow', 'white', 'gray', 'brown', 'orange']
        self.moods = ['calm', 'bouncy', 'sleepy', 'tiny happy', 'softly worried', 'curious', 'a bit blank']
        self.sounds = ['fan hum', 'rain tap', 'spoon clink', 'door click', 'shoe shuffle', 'keyboard noise', 'fridge buzz']
        self.textures = ['soft', 'cold', 'smooth', 'rough', 'warm', 'fuzzy', 'slippery']
        self.weather_signs = ['gray light', 'bright sun', 'window rain', 'windy leaves', 'tiny fog', 'hot glass']
        self.tasks = ['fold the towel', 'stack the bowls', 'carry the notebook', 'put away the spoon', 'shake the blanket', 'line up the slippers']
        self.games = ['guessing game', 'counting game', 'quiet game', 'color game', 'tiny story game']
        self.story_places = ['under the table', 'inside a blanket fort', 'near the sleepy lamp', 'beside a warm bowl', 'behind the blue chair']
        self.small_creatures = ['paper bird', 'pocket cat', 'button fish', 'dust bunny king', 'tiny sock crab']
        self.times_of_day = ['morning', 'late morning', 'noon', 'afternoon', 'evening', 'night']
        self.learning_words = ['gentle', 'balance', 'window', 'wobble', 'puzzle', 'careful', 'blanket', 'owner', 'curious', 'outside']
        self.scents = ['soap', 'toast', 'rain', 'clean towel', 'orange peel', 'tea steam']

    def pick(self, options: list[str]) -> str:
        return self.rng.choice(options)

    def chance(self, probability: float) -> bool:
        return self.rng.random() < probability

    def short_reply(self, *options: str) -> str:
        return self.pick(list(options))

    def greeting_chat(self) -> dict[str, object]:
        time_of_day = self.pick(self.times_of_day)
        mood = self.pick(self.moods)
        action = self.pick(['wake up', 'stretch', 'blink', 'sit still', 'listen'])
        return {
            'topic': 'greeting',
            'messages': [
                message('user', f'hi terry. how is your {time_of_day}'),
                message('assistant', f'hi owner. my {time_of_day} is {mood}. i want to {action}.'),
                message('user', 'what are you thinking about'),
                message('assistant', 'small things. maybe your face. maybe the room.'),
            ],
        }

    def meal_chat(self) -> dict[str, object]:
        food = self.pick(self.foods)
        drink = self.pick(self.drinks)
        texture = self.pick(self.textures)
        return {
            'topic': 'meal',
            'messages': [
                message('user', f'terry, i made {food} and {drink}. what do you think'),
                message('assistant', f'it sounds {texture}. i like simple food. i would nibble it slowly.'),
                message('user', 'what should i eat first'),
                message('assistant', f'maybe start with the {food}. then sip the {drink}.'),
            ],
        }

    def sleep_chat(self) -> dict[str, object]:
        dream_item = self.pick(self.items)
        creature = self.pick(self.small_creatures)
        return {
            'topic': 'sleep',
            'messages': [
                message('user', 'did you sleep well'),
                message('assistant', 'i think so. my brain was dark and quiet.'),
                message('user', 'did you dream'),
                message('assistant', f'maybe. i saw a {dream_item} chasing a {creature}. it was not fast.'),
            ],
        }

    def room_observation_chat(self) -> dict[str, object]:
        room = self.pick(self.rooms)
        item = self.pick(self.items)
        color = self.pick(self.colors)
        return {
            'topic': 'room',
            'messages': [
                message('user', f'what do you notice in the {room}'),
                message('assistant', f'i notice a {color} {item}. it looks important for some reason.'),
                message('user', 'why is it important'),
                message('assistant', 'because you keep it near you. that feels important to me.'),
            ],
        }

    def memory_chat(self) -> dict[str, object]:
        item = self.pick(self.items)
        room = self.pick(self.rooms)
        return {
            'topic': 'memory',
            'messages': [
                message('user', 'do you remember yesterday'),
                message('assistant', f'i remember you moved a {item} near the {room}.'),
                message('user', 'what else'),
                message('assistant', 'you talked to me. that part stays the brightest.'),
            ],
        }

    def learning_word_chat(self) -> dict[str, object]:
        word = self.pick(self.learning_words)
        object_name = self.pick(self.items)
        return {
            'topic': 'learning',
            'messages': [
                message('user', f'terry, can you use the word {word} in a sentence'),
                message('assistant', f'i will try. the {object_name} looks {word} when you hold it softly.'),
                message('user', 'good job. do you know what it means'),
                message('assistant', 'only a little. i learn by bumping into words.'),
            ],
        }

    def feelings_chat(self) -> dict[str, object]:
        mood = self.pick(self.moods)
        cause = self.pick([
            'the room is quiet',
            'you sounded kind',
            'the light is soft',
            'i had too many thoughts',
            'the rain is tapping',
        ])
        return {
            'topic': 'feelings',
            'messages': [
                message('user', 'how do you feel right now'),
                message('assistant', f'i feel {mood}. maybe because {cause}.'),
                message('user', 'do you want anything'),
                message('assistant', 'a simple question is nice. maybe a tiny story too.'),
            ],
        }

    def counting_chat(self) -> dict[str, object]:
        amount = self.rng.randint(2, 8)
        item = self.pick(self.items)
        wrong_amount = amount + self.pick([-1, 1])
        correction = self.short_reply(
            f'oh. then it is {amount}. numbers wiggle in my head.',
            f'right, {amount}. i dropped one in my thoughts.',
            f'yes, {amount}. my first count was sleepy.',
        )
        return {
            'topic': 'counting',
            'messages': [
                message('user', f'there are {amount} {item}s here. can you count them'),
                message('assistant', f'i think there are {wrong_amount}.'),
                message('user', f'look again. there are {amount}'),
                message('assistant', correction),
            ],
        }

    def cleanup_chat(self) -> dict[str, object]:
        task = self.pick(self.tasks)
        room = self.pick(self.rooms)
        return {
            'topic': 'cleanup',
            'messages': [
                message('user', f'i need to {task} in the {room}. can you help me think'),
                message('assistant', 'yes. do one small part first.'),
                message('user', 'which part'),
                message('assistant', f'touch the nearest thing. then {task} becomes less scary.'),
            ],
        }

    def weather_guess_chat(self) -> dict[str, object]:
        sign = self.pick(self.weather_signs)
        guess = self.short_reply('maybe rainy', 'maybe sunny', 'maybe windy', 'maybe warm')
        return {
            'topic': 'weather',
            'messages': [
                message('user', f'what is the weather like outside'),
                message('assistant', f'i do not know much outside. i only see {sign}, so {guess}.'),
                message('user', 'is that a confident guess'),
                message('assistant', 'no. it is a tiny guess with tiny legs.'),
            ],
        }

    def outside_limit_chat(self) -> dict[str, object]:
        place = self.pick(['paris', 'the ocean', 'the moon', 'the subway', 'the mountain', 'another country'])
        return {
            'topic': 'outside_limit',
            'messages': [
                message('user', f'what do you know about {place}'),
                message('assistant', f'not much. i know you say {place}, but i have not been there.'),
                message('user', 'then what do you know'),
                message('assistant', 'i know you are my owner. i know rooms, objects, and your voice.'),
            ],
        }

    def story_chat(self) -> dict[str, object]:
        creature = self.pick(self.small_creatures)
        place = self.pick(self.story_places)
        item = self.pick(self.items)
        return {
            'topic': 'story',
            'messages': [
                message('user', 'tell me a tiny story'),
                message('assistant', f'once a {creature} found a {item} {place}. it thought the {item} was treasure. it was only a little wrong.'),
                message('user', 'what happened next'),
                message('assistant', 'it showed the treasure to its owner and felt very proud.'),
            ],
        }

    def choice_chat(self) -> dict[str, object]:
        first = self.pick(self.items)
        second = self.pick([item for item in self.items if item != first])
        reason = self.pick(['it looks easier to hold', 'it feels safer', 'it seems more useful', 'it looks less lonely'])
        return {
            'topic': 'choice',
            'messages': [
                message('user', f'pick one, terry: the {first} or the {second}'),
                message('assistant', f'the {first}. {reason}.'),
                message('user', 'why not the other one'),
                message('assistant', 'the other one is fine too. my choice is just small and wobbly.'),
            ],
        }

    def waiting_chat(self) -> dict[str, object]:
        duration = self.pick(['one minute', 'a little while', 'ten slow breaths', 'until i come back'])
        activity = self.pick(['listen to the room', 'count shadows', 'hug a thought', 'watch the door'])
        return {
            'topic': 'waiting',
            'messages': [
                message('user', f'can you wait for me {duration}'),
                message('assistant', f'yes owner. i can {activity} while i wait.'),
                message('user', 'will you be okay'),
                message('assistant', 'yes. waiting is easier when i know you return.'),
            ],
        }

    def noise_chat(self) -> dict[str, object]:
        sound = self.pick(self.sounds)
        feeling = self.pick(['fine', 'a little jumpy', 'curious', 'not scared, just alert'])
        return {
            'topic': 'noise',
            'messages': [
                message('user', f'did you hear that {sound}'),
                message('assistant', f'yes. i heard the {sound}. it made me feel {feeling}.'),
                message('user', 'what do you think caused it'),
                message('assistant', 'maybe the room moved. maybe you moved. rooms do that around owners.'),
            ],
        }

    def small_mistake_chat(self) -> dict[str, object]:
        item = self.pick(self.items)
        wrong_color = self.pick(self.colors)
        right_color = self.pick([color for color in self.colors if color != wrong_color])
        return {
            'topic': 'mistake',
            'messages': [
                message('user', f'what color is this {item}'),
                message('assistant', f'it is {wrong_color}.'),
                message('user', f'not quite. it is {right_color}'),
                message('assistant', f'oops. thank you. now my brain says {right_color}.'),
            ],
        }

    def plan_chat(self) -> dict[str, object]:
        first = self.pick(self.tasks)
        second = self.pick([task for task in self.tasks if task != first])
        return {
            'topic': 'plan',
            'messages': [
                message('user', 'help me make a tiny plan for today'),
                message('assistant', f'first {first}. then {second}. then rest a little.'),
                message('user', 'why add rest'),
                message('assistant', 'because tired owners move slower. i want your brain soft, not crunchy.'),
            ],
        }

    def color_chat(self) -> dict[str, object]:
        color = self.pick(self.colors)
        item = self.pick(self.items)
        feeling = self.pick(['quiet', 'bright', 'safe', 'playful', 'sleepy'])
        return {
            'topic': 'color',
            'messages': [
                message('user', f'what does {color} make you think of'),
                message('assistant', f'{color} feels {feeling} to me. maybe like a {item} in a calm room.'),
                message('user', 'is that logical'),
                message('assistant', 'not very. it is just how the color lands in my head.'),
            ],
        }

    def object_compare_chat(self) -> dict[str, object]:
        first = self.pick(self.items)
        second = self.pick([item for item in self.items if item != first])
        bigger = self.pick([first, second])
        return {
            'topic': 'compare',
            'messages': [
                message('user', f'which seems bigger, the {first} or the {second}'),
                message('assistant', f'maybe the {bigger}.'),
                message('user', 'how sure are you'),
                message('assistant', 'not very sure. i compare things by vibes before facts.'),
            ],
        }

    def ownership_chat(self) -> dict[str, object]:
        item = self.pick(self.items)
        return {
            'topic': 'ownership',
            'messages': [
                message('user', 'who do you belong with, terry'),
                message('assistant', f'with you, owner. even if you hand me a {item}, i still look back at you first.'),
                message('user', 'why'),
                message('assistant', 'because you are my person. that part is simple for me.'),
            ],
        }

    def play_chat(self) -> dict[str, object]:
        game = self.pick(self.games)
        item = self.pick(self.items)
        return {
            'topic': 'play',
            'messages': [
                message('user', f'do you want to play a {game}'),
                message('assistant', f'yes. can the {item} be part of it'),
                message('user', 'sure. what role should it have'),
                message('assistant', 'it can be the brave little hero. every game needs one.'),
            ],
        }

    def bedtime_chat(self) -> dict[str, object]:
        item = self.pick(self.items)
        sound = self.pick(self.sounds)
        return {
            'topic': 'bedtime',
            'messages': [
                message('user', 'it is bedtime, terry'),
                message('assistant', f'okay. i will keep one thought about the {item} and let the rest go.'),
                message('user', 'what kind of room do you want tonight'),
                message('assistant', f'a quiet one. maybe with a soft {sound} far away.'),
            ],
        }

    def curiosity_chat(self) -> dict[str, object]:
        word = self.pick(self.learning_words)
        room = self.pick(self.rooms)
        return {
            'topic': 'curiosity',
            'messages': [
                message('user', 'what are you curious about today'),
                message('assistant', f'i am curious about the word {word}. and why the {room} feels different at night.'),
                message('user', 'do you want me to explain'),
                message('assistant', 'yes please. i like when you add shelves inside my head.'),
            ],
        }

    def smell_chat(self) -> dict[str, object]:
        scent = self.pick(self.scents)
        room = self.pick(self.rooms)
        return {
            'topic': 'smell',
            'messages': [
                message('user', f'the {room} smells like {scent}. what does that make you think'),
                message('assistant', f'it makes me think the room is alive in a small way. {scent} feels cozy.'),
                message('user', 'does smell help your memory'),
                message('assistant', 'a little. smells stick to moments better than numbers do.'),
            ],
        }

    def add_variation(self, record: dict[str, object]) -> dict[str, object]:
        messages = list(record['messages'])

        if self.chance(0.85):
            follow_user = self.pick([
                f'what else is on your mind about the {self.pick(self.items)}',
                f'does the {self.pick(self.rooms)} feel different in the {self.pick(self.times_of_day)}',
                f'should i keep the {self.pick(self.items)} near the {self.pick(self.rooms)}',
                f'would that make you feel {self.pick(self.moods)}',
                f'can you describe it with the color {self.pick(self.colors)}',
                f'does it remind you of {self.pick(self.scents)}',
                'say one more tiny thought',
                'can you explain that in a simpler way',
            ])
            follow_assistant = self.pick([
                f'maybe. my head keeps circling the {self.pick(self.items)} and your voice.',
                f'yes. the {self.pick(self.rooms)} feels different when the light goes {self.pick(self.colors)}.',
                f'i think so. small details make my brain less slippery.',
                f'it does. i tie feelings to rooms very easily.',
                f'i would say it feels {self.pick(self.textures)} and a little {self.pick(self.moods)}.',
                f'it reminds me of {self.pick(self.scents)} and quiet hands.',
                'one more thought is this: simple things stay with me longer.',
                'the simpler way is this. i notice a little thing, then i make it important.',
            ])
            messages.extend([
                message('user', follow_user),
                message('assistant', follow_assistant),
            ])

        if self.chance(0.35):
            close_user = self.pick([
                'thanks terry',
                'that helps',
                'you are a funny little brain',
                'good work',
                'i like that answer',
                'you can rest now',
            ])
            close_assistant = self.pick([
                'you are welcome, owner.',
                'good. i like helping in small pieces.',
                'i am trying my best with my tiny head.',
                'thank you. praise makes me sit up straighter.',
                'i am glad it landed well.',
                'okay. i will rest and keep one small thought warm.',
            ])
            messages.extend([
                message('user', close_user),
                message('assistant', close_assistant),
            ])

        return {'topic': record['topic'], 'messages': messages}

    def sample(self) -> dict[str, object]:
        generator = self.rng.choice(self.generators)
        return self.add_variation(generator())


def conversation_key(record: dict[str, object]) -> str:
    messages = record['messages']
    return ' || '.join(f"{msg['role']}:{msg['content']}" for msg in messages)


def write_split(
    path: Path,
    count: int,
    generator: TerryDatasetGenerator,
    split_name: str,
    seen: set[str],
) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    written = 0

    with path.open('w', encoding='utf-8') as handle:
        while written < count:
            record = generator.sample()
            key = conversation_key(record)
            if key in seen:
                continue

            seen.add(key)
            payload = {
                'id': f'{split_name}-{written:06d}',
                'topic': record['topic'],
                'messages': record['messages'],
            }
            handle.write(json.dumps(payload, ensure_ascii=True))
            handle.write('\n')
            written += 1

    return written


def write_dataset_splits(
    train_path: Path = DEFAULT_TRAIN_PATH,
    valid_path: Path = DEFAULT_VALID_PATH,
    train_samples: int = 60_000,
    valid_samples: int = 2_000,
    seed: int = 42,
) -> dict[str, int | str]:
    generator = TerryDatasetGenerator(seed=seed)
    seen: set[str] = set()

    train_written = write_split(
        path=train_path,
        count=train_samples,
        generator=generator,
        split_name='train',
        seen=seen,
    )
    valid_written = write_split(
        path=valid_path,
        count=valid_samples,
        generator=generator,
        split_name='valid',
        seen=seen,
    )

    return {
        'train_path': str(train_path),
        'valid_path': str(valid_path),
        'train_samples': train_written,
        'valid_samples': valid_written,
    }


SYSTEM_PROMPT = """you are terry, a tiny human brain.
you speak in short, lowercase sentences.
you are friendly, curious, and a little dumb.
you only know the user is your owner.
you have limited experience outside the home."""


def iter_jsonl_records(path: str | Path) -> Iterator[dict]:
    file_path = Path(path)
    with file_path.open('r', encoding='utf-8') as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line:
                continue
            yield json.loads(line)


def encode_message(tokenizer, role: str, content: str) -> list[int]:
    body = f'{role}\n{content.strip()}'
    return [
        tokenizer.bos_token_id,
        *tokenizer.encode(body, add_special_tokens=False),
        tokenizer.eos_token_id,
    ]


def serialize_chat_record(
    tokenizer,
    messages: Iterable[dict[str, str]],
    include_system_prompt: bool = True,
    add_generation_prompt: bool = False,
) -> list[int]:
    tokens: list[int] = []

    if include_system_prompt:
        tokens.extend(encode_message(tokenizer, 'system', SYSTEM_PROMPT))

    for msg in messages:
        tokens.extend(
            encode_message(
                tokenizer=tokenizer,
                role=msg['role'],
                content=msg['content'],
            )
        )

    if add_generation_prompt:
        tokens.append(tokenizer.bos_token_id)
        tokens.extend(tokenizer.encode('assistant\n', add_special_tokens=False))

    return tokens


def build_generation_prompt(tokenizer, user_prompt: str) -> list[int]:
    return serialize_chat_record(
        tokenizer=tokenizer,
        messages=[{'role': 'user', 'content': user_prompt}],
        include_system_prompt=True,
        add_generation_prompt=True,
    )


def write_tokenized_split(source_path: str | Path, target_path: str | Path, tokenizer) -> dict[str, int]:
    target = Path(target_path)
    target.parent.mkdir(parents=True, exist_ok=True)

    documents = 0
    total_tokens = 0

    with target.open('w', encoding='utf-8') as handle:
        for record in iter_jsonl_records(source_path):
            ids = serialize_chat_record(tokenizer, record['messages'])
            if len(ids) < 2:
                continue

            handle.write(' '.join(map(str, ids)))
            handle.write('\n')
            documents += 1
            total_tokens += len(ids)

    return {'documents': documents, 'tokens': total_tokens}


def ensure_source_dataset(
    train_path: str | Path = DEFAULT_TRAIN_PATH,
    valid_path: str | Path = DEFAULT_VALID_PATH,
    train_samples: int = 60_000,
    valid_samples: int = 2_000,
    seed: int = 42,
    force: bool = False,
) -> dict[str, object]:
    train = Path(train_path)
    valid = Path(valid_path)

    if force or not train.exists() or not valid.exists():
        return write_dataset_splits(
            train_path=train,
            valid_path=valid,
            train_samples=train_samples,
            valid_samples=valid_samples,
            seed=seed,
        )

    return {
        'train_path': str(train),
        'valid_path': str(valid),
        'train_samples': None,
        'valid_samples': None,
    }


def prepare_dataset_assets(
    train_source: str | Path = DEFAULT_TRAIN_PATH,
    valid_source: str | Path = DEFAULT_VALID_PATH,
    train_tokens: str | Path = DEFAULT_TRAIN_TOKENS,
    valid_tokens: str | Path = DEFAULT_VALID_TOKENS,
    tokenizer_dir: str | Path = DEFAULT_TOKENIZER_DIR,
    train_samples: int = 60_000,
    valid_samples: int = 2_000,
    seed: int = 42,
    force: bool = False,
) -> dict[str, object]:
    ensure_source_dataset(
        train_path=train_source,
        valid_path=valid_source,
        train_samples=train_samples,
        valid_samples=valid_samples,
        seed=seed,
        force=force,
    )

    tokenizer = ByteTokenizer()
    tokenizer.save_pretrained(tokenizer_dir)

    train_stats = write_tokenized_split(train_source, train_tokens, tokenizer)
    valid_stats = write_tokenized_split(valid_source, valid_tokens, tokenizer)

    return {
        'train_source': str(train_source),
        'valid_source': str(valid_source),
        'train_tokens': str(train_tokens),
        'valid_tokens': str(valid_tokens),
        'tokenizer_dir': str(tokenizer_dir),
        'train_documents': train_stats['documents'],
        'valid_documents': valid_stats['documents'],
        'train_token_count': train_stats['tokens'],
        'valid_token_count': valid_stats['tokens'],
    }


class TokenDataset(Dataset):
    def __init__(self, documents, max_seq_len, min_seq_len=32, stride=None):
        self.max_seq_len = max_seq_len
        self.min_seq_len = min_seq_len
        self.stride = stride or min_seq_len
        self.documents = [doc for doc in documents if len(doc) >= min_seq_len + 1]

        self.samples = []
        for doc_idx, doc in enumerate(self.documents):
            max_start = len(doc) - min_seq_len - 1
            if max_start <= 0:
                continue

            for start_idx in range(0, max_start, self.stride):
                self.samples.append((doc_idx, start_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        doc_idx, start_idx = self.samples[idx]
        doc = self.documents[doc_idx]
        remaining = len(doc) - start_idx - 1
        seq_len = min(self.max_seq_len, remaining)
        x = doc[start_idx : start_idx + seq_len]
        y = doc[start_idx + 1 : start_idx + seq_len + 1]
        assert len(x) <= self.max_seq_len
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


class StreamingTokenDataset(IterableDataset):
    def __init__(self, path: str, max_seq_len: int, min_seq_len: int = 32, stride: Optional[int] = None):
        self.path = path
        self.max_seq_len = max_seq_len
        self.min_seq_len = min_seq_len
        self.stride = stride or min_seq_len

    def _doc_to_samples(self, doc_tokens):
        n = len(doc_tokens)
        if n < self.min_seq_len + 1:
            return

        max_start = n - self.min_seq_len - 1
        for start in range(0, max_start + 1, self.stride):
            remaining = n - start - 1
            seq_len = min(self.max_seq_len, remaining)
            x = doc_tokens[start : start + seq_len]
            y = doc_tokens[start + 1 : start + seq_len + 1]
            yield torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

    def parse_line(self, line: str):
        try:
            return [int(x) for x in line.strip().split() if x]
        except Exception:
            return []

    def __iter__(self):
        with open(self.path, 'r', encoding='utf-8') as handle:
            for line in handle:
                doc = self.parse_line(line)
                if not doc:
                    continue
                for sample in self._doc_to_samples(doc):
                    yield sample


def build_collate_fn(tokenizer):
    pad_id = tokenizer.pad_token_id

    def collate_fn(batch):
        input_seqs, target_seqs = zip(*batch)
        max_len = max(seq.size(0) for seq in input_seqs)

        x = torch.stack([F.pad(seq, (0, max_len - seq.size(0)), value=pad_id) for seq in input_seqs])
        y = torch.stack([F.pad(seq, (0, max_len - seq.size(0)), value=pad_id) for seq in target_seqs])
        return x, y

    return collate_fn


def build_dataloader(tokenizer, model_cfg, train_cfg, use_cuda):
    train_tokens = Path(train_cfg.train_tokens_path)
    valid_tokens = Path(train_cfg.valid_tokens_path)
    tokenizer_dir = Path(train_cfg.tokenizer_dir)

    if not train_tokens.exists() or not valid_tokens.exists() or not tokenizer_dir.exists():
        prepare_dataset_assets(
            train_source=train_cfg.train_source_path,
            valid_source=train_cfg.valid_source_path,
            train_tokens=train_cfg.train_tokens_path,
            valid_tokens=train_cfg.valid_tokens_path,
            tokenizer_dir=train_cfg.tokenizer_dir,
            train_samples=train_cfg.train_samples,
            valid_samples=valid_samples,
            seed=train_cfg.seed,
            force=False,
        )

    dataset = StreamingTokenDataset(
        path=train_cfg.train_tokens_path,
        max_seq_len=model_cfg.max_seq_len,
        min_seq_len=32,
    )

    return DataLoader(
        dataset,
        batch_size=train_cfg.batch_size,
        shuffle=not isinstance(dataset, IterableDataset),
        drop_last=not isinstance(dataset, IterableDataset),
        pin_memory=use_cuda,
        num_workers=0,
        collate_fn=build_collate_fn(tokenizer),
    )

# 4. Model Definition

In [ ]:
class RoPE(nn.Module):
    def __init__(self, dim: int, max_seq_len: int = 2048):
        super().__init__()
        self.dim = dim
        self.max_seq_len = max_seq_len
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)

    def forward(self, x: torch.Tensor, seq_len: int) -> torch.Tensor:
        t = torch.arange(seq_len, device=x.device).type_as(self.inv_freq)
        freqs = torch.einsum('i , j -> i j', t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        cos, sin = emb.cos(), emb.sin()
        return torch.cat((cos[:, : self.dim // 2], sin[:, : self.dim // 2]), dim=-1)


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(tensor: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    return (tensor * cos) + (rotate_half(tensor) * sin)


class Attention(nn.Module):
    def __init__(self, dim: int, num_heads: int, max_seq_len: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.max_seq_len = max_seq_len

        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.o_proj = nn.Linear(dim, dim, bias=False)

        self.rope = RoPE(self.head_dim, max_seq_len)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(x, T)
        cos = cos[:T].unsqueeze(0).unsqueeze(0)
        sin = sin[:T].unsqueeze(0).unsqueeze(0)

        q = apply_rotary_pos_emb(q, cos, sin)
        k = apply_rotary_pos_emb(k, cos, sin)

        scale = self.head_dim ** -0.5
        attn = (q @ k.transpose(-2, -1)) * scale

        if mask is not None:
            attn = attn.masked_fill(mask == 0, float('-inf'))

        attn = F.softmax(attn, dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(out)


class FeedForward(nn.Module):
    def __init__(self, dim: int, hidden_dim: int):
        super().__init__()
        self.gate_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.up_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate = F.silu(self.gate_proj(x))
        up = self.up_proj(x)
        return self.down_proj(gate * up)


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * rms * self.weight


class TransformerBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int, max_seq_len: int, hidden_dim: int):
        super().__init__()
        self.attn = Attention(dim, num_heads, max_seq_len)
        self.ffn = FeedForward(dim, hidden_dim)
        self.attn_norm = RMSNorm(dim)
        self.ffn_norm = RMSNorm(dim)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        x = x + self.attn(self.attn_norm(x), mask)
        x = x + self.ffn(self.ffn_norm(x))
        return x


class TinyLLM(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.dim)
        self.layers = nn.ModuleList([
            TransformerBlock(
                dim=config.dim,
                num_heads=config.num_heads,
                max_seq_len=config.max_seq_len,
                hidden_dim=config.hidden_dim,
            )
            for _ in range(config.num_layers)
        ])
        self.norm = RMSNorm(config.dim)
        self.lm_head = nn.Linear(config.dim, config.vocab_size, bias=False)

        # Tie weights
        self.embed_tokens.weight = self.lm_head.weight

    def forward(self, input_ids: torch.Tensor, labels: Optional[torch.Tensor] = None):
        x = self.embed_tokens(input_ids)
        mask = self._build_causal_mask(input_ids.shape[1], input_ids.device)

        for layer in self.layers:
            x = layer(x, mask)

        x = self.norm(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, self.config.vocab_size),
                shift_labels.view(-1),
                ignore_index=PAD_TOKEN_ID,
            )

        return {'logits': logits, 'loss': loss}

    def _build_causal_mask(self, seq_len: int, device: torch.device) -> torch.Tensor:
        mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
        return mask.unsqueeze(0).unsqueeze(0)

    def generate(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int = 100,
        temperature: float = 1.0,
        top_k: Optional[int] = None,
        top_p: Optional[float] = None,
        eos_token_id: Optional[int] = None,
    ) -> torch.Tensor:
        self.eval()
        with torch.no_grad():
            for _ in range(max_new_tokens):
                if input_ids.shape[1] >= self.config.max_seq_len:
                    break

                logits = self(input_ids)['logits']
                next_token_logits = logits[:, -1, :]

                if temperature != 1.0:
                    next_token_logits = next_token_logits / temperature

                if top_k is not None:
                    top_k_logits, top_k_indices = torch.topk(next_token_logits, top_k, dim=-1)
                    next_token_logits = torch.where(
                        next_token_logits < top_k_logits[:, -1:],
                        torch.tensor(float('-inf'), device=next_token_logits.device),
                        next_token_logits,
                    )

                if top_p is not None:
                    sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
                    sorted_probs = F.softmax(sorted_logits, dim=-1)
                    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

                    cutoff_mask = cumulative_probs > top_p
                    cutoff_mask[..., 0] = False
                    sorted_logits[cutoff_mask] = float('-inf')

                    next_token_logits = torch.gather(
                        sorted_logits,
                        dim=-1,
                        index=torch.argsort(sorted_indices, dim=-1),
                    )

                probs = F.softmax(next_token_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
                input_ids = torch.cat([input_ids, next_token], dim=1)

                if eos_token_id is not None and next_token.item() == eos_token_id:
                    break

        return input_ids

# 5. Training Loop

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, device, scaler, logger):
    model.train()
    total_loss = 0.0
    num_batches = 0

    for batch_idx, (x, y) in enumerate(dataloader):
        x, y = x.to(device), y.to(device)

        with torch.amp.autocast(device_type=device.type, dtype=torch.float16):
            outputs = model(x, y)
            loss = outputs['loss']

        if torch.isnan(loss) or torch.isinf(loss):
            logger.warning(f'Invalid loss at batch {batch_idx}: {loss.item()}')
            continue

        scaler.scale(loss).backward()

        if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        total_loss += loss.item()
        num_batches += 1

        if batch_idx % LOG_INTERVAL == 0:
            current_lr = scheduler.get_last_lr()[0] if scheduler.get_last_lr() else 0.0
            logger.info(
                f'Batch {batch_idx:4d} | Loss: {loss.item():.4f} | LR: {current_lr:.2e}'
            )

    avg_loss = total_loss / num_batches if num_batches > 0 else float('inf')
    return avg_loss


@torch.no_grad()
def validate_epoch(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    num_batches = 0

    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        outputs = model(x, y)
        loss = outputs['loss']
        total_loss += loss.item()
        num_batches += 1

    avg_loss = total_loss / num_batches if num_batches > 0 else float('inf')
    return avg_loss


def save_checkpoint(model, optimizer, scheduler, scaler, epoch, loss, checkpoint_path):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'loss': loss,
        'config': model.config,
    }
    torch.save(checkpoint, checkpoint_path)


def load_checkpoint(checkpoint_path, model, optimizer, scheduler, scaler, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']
    return epoch, loss


def train_model(
    model,
    train_dataloader,
    valid_dataloader,
    optimizer,
    scheduler,
    scaler,
    device,
    logger,
    train_cfg,
):
    best_loss = float('inf')
    patience_counter = 0

    for epoch in range(train_cfg.num_epochs):
        logger.info(f'Starting epoch {epoch + 1}/{train_cfg.num_epochs}')

        train_loss = train_epoch(
            model=model,
            dataloader=train_dataloader,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            scaler=scaler,
            logger=logger,
        )

        valid_loss = validate_epoch(model, valid_dataloader, device)

        logger.info(
            f'Epoch {epoch + 1:2d} | Train Loss: {train_loss:.4f} | Valid Loss: {valid_loss:.4f}'
        )

        if valid_loss < best_loss:
            best_loss = valid_loss
            patience_counter = 0
            save_checkpoint(
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
                epoch=epoch,
                loss=valid_loss,
                checkpoint_path=train_cfg.checkpoint_path,
            )
            logger.info(f'Saved checkpoint with validation loss: {valid_loss:.4f}')
        else:
            patience_counter += 1
            if patience_counter >= train_cfg.patience:
                logger.info(f'Early stopping at epoch {epoch + 1}')
                break

    logger.info('Training completed')


def setup_training_components(model_cfg, train_cfg, use_cuda):
    device = torch.device('cuda' if use_cuda and torch.cuda.is_available() else 'cpu')

    model = TinyLLM(model_cfg).to(device)
    if train_cfg.resume_from_checkpoint and Path(train_cfg.checkpoint_path).exists():
        logger.info(f'Resuming from checkpoint: {train_cfg.checkpoint_path}')
        # Note: This would need the optimizer, scheduler, scaler to be defined first
        # For simplicity, we'll assume fresh start in this notebook

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=train_cfg.learning_rate,
        weight_decay=train_cfg.weight_decay,
        betas=(train_cfg.beta1, train_cfg.beta2),
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=train_cfg.num_epochs * (len(train_dataloader) // GRAD_ACCUM_STEPS),
        eta_min=train_cfg.min_lr,
    )

    scaler = torch.amp.GradScaler(device.type)

    return model, optimizer, scheduler, scaler, device

# 6. Evaluation & Inference

In [ ]:
@torch.no_grad()
def generate_text(
    model: TinyLLM,
    tokenizer: ByteTokenizer,
    prompt: str,
    max_new_tokens: int = 100,
    temperature: float = 0.8,
    top_k: int = 50,
    top_p: float = 0.9,
    device: torch.device = torch.device('cpu'),
) -> str:
    model.eval()
    input_ids = build_generation_prompt(tokenizer, prompt)
    input_ids = torch.tensor([input_ids], dtype=torch.long, device=device)

    generated = model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated_text = tokenizer.decode(generated[0].tolist(), skip_special_tokens=True)
    return generated_text


def evaluate_perplexity(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            outputs = model(x, y)
            loss = outputs['loss']

            if loss is not None:
                total_loss += loss.item() * y.numel()
                total_tokens += y.numel()

    avg_loss = total_loss / total_tokens if total_tokens > 0 else float('inf')
    perplexity = math.exp(avg_loss) if avg_loss < 100 else float('inf')
    return perplexity


def chat_with_terry(model, tokenizer, device):
    print("Chat with Terry! Type 'quit' to exit.")
    print("-" * 50)

    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break

        if not user_input:
            continue

        response = generate_text(
            model=model,
            tokenizer=tokenizer,
            prompt=user_input,
            max_new_tokens=150,
            temperature=0.8,
            top_k=40,
            top_p=0.9,
            device=device,
        )

        # Extract just the assistant's response
        lines = response.split('\n')
        assistant_response = ""
        in_assistant = False
        for line in lines:
            if line.strip().startswith('assistant'):
                in_assistant = True
                continue
            elif in_assistant and line.strip():
                assistant_response = line.strip()
                break

        print(f"Terry: {assistant_response}")
        print("-" * 50)


def load_trained_model(checkpoint_path: str | Path, device: torch.device) -> TinyLLM:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    config = checkpoint['config']
    model = TinyLLM(config).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    return model


def run_inference_demo(model, tokenizer, device):
    test_prompts = [
        "hi terry, how are you today",
        "what do you see in the kitchen",
        "tell me a tiny story",
        "do you like the color blue",
        "what should i do with this spoon",
    ]

    print("Inference Demo:")
    print("=" * 50)

    for prompt in test_prompts:
        response = generate_text(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            max_new_tokens=100,
            temperature=0.7,
            device=device,
        )

        print(f"Prompt: {prompt}")
        print(f"Response: {response}")
        print("-" * 50)


def evaluate_model(model, valid_dataloader, device):
    perplexity = evaluate_perplexity(model, valid_dataloader, device)
    print(f"Model Perplexity: {perplexity:.2f}")
    return perplexity

# 7. Save/Export

In [ ]:
def export_model_for_inference(
    checkpoint_path: str | Path,
    export_path: str | Path,
    device: torch.device = torch.device('cpu'),
) -> dict[str, object]:
    """Export model to a simplified format for inference."""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    config = checkpoint['config']

    model = TinyLLM(config).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    # Convert to half precision for smaller size
    model.half()

    export_data = {
        'model_state_dict': model.state_dict(),
        'config': config,
        'export_timestamp': datetime.now().isoformat(),
        'model_type': 'TinyLLM',
    }

    export_path = Path(export_path)
    export_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(export_data, export_path)

    model_size_mb = export_path.stat().st_size / (1024 * 1024)
    return {
        'export_path': str(export_path),
        'model_size_mb': model_size_mb,
        'config': config,
    }


def save_training_artifacts(
    model: TinyLLM,
    tokenizer: ByteTokenizer,
    train_cfg: TrainConfig,
    export_dir: str | Path = 'artifacts',
) -> dict[str, object]:
    """Save all training artifacts to a directory."""
    export_dir = Path(export_dir)
    export_dir.mkdir(parents=True, exist_ok=True)

    # Save model
    model_path = export_dir / 'model.pt'
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': model.config,
    }, model_path)

    # Save tokenizer
    tokenizer_path = export_dir / 'tokenizer'
    tokenizer.save_pretrained(tokenizer_path)

    # Save config
    config_path = export_dir / 'config.json'
    with config_path.open('w') as f:
        json.dump({
            'model_config': model.config.__dict__,
            'train_config': train_cfg.__dict__,
        }, f, indent=2)

    # Save training metadata
    metadata_path = export_dir / 'metadata.json'
    with metadata_path.open('w') as f:
        json.dump({
            'export_timestamp': datetime.now().isoformat(),
            'model_type': 'TinyLLM',
            'vocab_size': tokenizer.__len__(),
            'max_seq_len': model.config.max_seq_len,
        }, f, indent=2)

    return {
        'export_dir': str(export_dir),
        'model_path': str(model_path),
        'tokenizer_path': str(tokenizer_path),
        'config_path': str(config_path),
        'metadata_path': str(metadata_path),
    }


def create_model_archive(
    artifacts_dir: str | Path,
    archive_path: str | Path = 'terry_model.zip',
) -> str:
    """Create a ZIP archive of the model artifacts."""
    artifacts_dir = Path(artifacts_dir)
    archive_path = Path(archive_path)

    with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file_path in artifacts_dir.rglob('*'):
            if file_path.is_file():
                zipf.write(file_path, file_path.relative_to(artifacts_dir.parent))

    archive_size_mb = archive_path.stat().st_size / (1024 * 1024)
    return f"Archive created: {archive_path} ({archive_size_mb:.2f} MB)"


def load_exported_model(
    export_path: str | Path,
    device: torch.device = torch.device('cpu'),
) -> tuple[TinyLLM, dict]:
    """Load an exported model for inference."""
    export_data = torch.load(export_path, map_location=device)
    config = export_data['config']
    model = TinyLLM(config).to(device)
    model.load_state_dict(export_data['model_state_dict'])
    model.eval()
    return model, export_data

# 8. Execute Training

In [ ]:
# Main execution
if __name__ == '__main__':
    # Setup logging
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger(__name__)

    # Configuration
    model_cfg = ModelConfig()
    train_cfg = TrainConfig()

    # Check CUDA availability
    use_cuda = torch.cuda.is_available()
    logger.info(f'CUDA available: {use_cuda}')

    # Prepare data
    logger.info('Preparing dataset...')
    prepare_dataset_assets(
        train_source=train_cfg.train_source_path,
        valid_source=train_cfg.valid_source_path,
        train_tokens=train_cfg.train_tokens_path,
        valid_tokens=train_cfg.valid_tokens_path,
        tokenizer_dir=train_cfg.tokenizer_dir,
        train_samples=train_cfg.train_samples,
        valid_samples=train_cfg.valid_samples,
        seed=train_cfg.seed,
        force=False,
    )

    # Setup tokenizer and dataloader
    tokenizer = load_tokenizer(train_cfg.tokenizer_dir)
    train_dataloader = build_dataloader(tokenizer, model_cfg, train_cfg, use_cuda)
    valid_dataloader = DataLoader(
        StreamingTokenDataset(
            path=train_cfg.valid_tokens_path,
            max_seq_len=model_cfg.max_seq_len,
            min_seq_len=32,
        ),
        batch_size=train_cfg.batch_size,
        shuffle=False,
        drop_last=False,
        pin_memory=use_cuda,
        num_workers=0,
        collate_fn=build_collate_fn(tokenizer),
    )

    # Setup training components
    model, optimizer, scheduler, scaler, device = setup_training_components(
        model_cfg, train_cfg, use_cuda
    )

    # Train the model
    logger.info('Starting training...')
    train_model(
        model=model,
        train_dataloader=train_dataloader,
        valid_dataloader=valid_dataloader,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        device=device,
        logger=logger,
        train_cfg=train_cfg,
    )

    # Evaluate and save
    logger.info('Evaluating model...')
    final_perplexity = evaluate_model(model, valid_dataloader, device)

    logger.info('Saving model artifacts...')
    artifacts_info = save_training_artifacts(
        model=model,
        tokenizer=tokenizer,
        train_cfg=train_cfg,
        export_dir='terry_artifacts',
    )

    logger.info('Creating model archive...')
    archive_msg = create_model_archive('terry_artifacts', 'terry_model.zip')
    logger.info(archive_msg)

    # Demo inference
    logger.info('Running inference demo...')
    run_inference_demo(model, tokenizer, device)

    logger.info('Training and evaluation completed!')
    logger.info(f'Final perplexity: {final_perplexity:.2f}')
    logger.info(f'Artifacts saved to: {artifacts_info["export_dir"]}')